In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("sesion11-bi-ml-distribuido")
    .master("local[*]")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
spark

In [3]:
import os

# Ubicación del parquet generado en S09
RUTA_EVENTOS_S9 = "./artifacts/sensores_observabilidad"

if not os.path.exists(RUTA_EVENTOS_S9):
    # Ruta alternativa en caso de estar en la carpeta previa
    RUTA_EVENTOS_S9 = "../s09-observabilidad-pipeline/artifacts/sensores_observabilidad"

eventos = spark.read.parquet(RUTA_EVENTOS_S9)
print(f"Filas eventos (S09): {eventos.count():,}")
eventos.select("sensorId", "temperatura", "isValid", "latencyMs").show(5)

Filas eventos (S09): 481
+--------------------+-----------+-------+---------+
|            sensorId|temperatura|isValid|latencyMs|
+--------------------+-----------+-------+---------+
|    esp32-fisico-lab|       13.7|   true|     4402|
|esp32-estacion-norte|       14.9|   true|     4401|
|  esp32-estacion-sur|       14.2|   true|     4401|
|    esp32-fisico-lab|       19.4|   true|     3483|
|    esp32-fisico-lab|       15.9|   true|     1397|
+--------------------+-----------+-------+---------+
only showing top 5 rows


In [4]:
import urllib.request
import json

# 1. Agregación analítica de negocio (KPIs)
kpi_eventos = eventos.agg(
    F.count("*").alias("total"),
    F.avg(F.col("isValid").cast("int")).alias("ratioValidos"),
    F.avg("latencyMs").alias("avgLatencyMs"),
).first()

print(f"Eventos procesados:   {kpi_eventos['total']}")
print(f"Ratio de validos:     {kpi_eventos['ratioValidos']:.4f}")
print(f"Latencia promedio:    {kpi_eventos['avgLatencyMs']:.2f} ms")

# 2. Formato de métricas para Prometheus (Gauge)
metricas_flujo = {
    "lambda26_bi_eventos_total": kpi_eventos["total"],
    "lambda26_bi_eventos_validos_ratio": kpi_eventos["ratioValidos"],
    "lambda26_bi_latencia_promedio_ms": kpi_eventos["avgLatencyMs"],
}

lineas = []
for nombre, valor in metricas_flujo.items():
    lineas.append(f"# TYPE {nombre} gauge")
    lineas.append(f"{nombre} {valor}")
cuerpo = "\n".join(lineas) + "\n"

# 3. Envío al Pushgateway
PUSHGATEWAY_FLUJO_URL = "http://pushgateway:9091/metrics/job/lambda26_bi_flujo/instance/s11_batch"
peticion = urllib.request.Request(PUSHGATEWAY_FLUJO_URL, data=cuerpo.encode("utf-8"), method="PUT")
with urllib.request.urlopen(peticion) as respuesta:
    print("\nPushgateway respondio HTTP Status:", respuesta.status)

Eventos procesados:   481
Ratio de validos:     1.0000
Latencia promedio:    2271.72 ms

Pushgateway respondio HTTP Status: 200


In [5]:
import time

time.sleep(3)  # Pequeña pausa para asegurar el scrape

for nombre in metricas_flujo:
    url = f"http://prometheus:9090/api/v1/query?query={nombre}"
    try:
        resultado = json.load(urllib.request.urlopen(url))
        valores = resultado["data"]["result"]
        if valores:
            print(f"Prometheus -> {nombre}: {valores[0]['value'][1]}")
        else:
            print(f"Prometheus -> {nombre}: SIN DATOS (esperar ciclo de scrape)")
    except Exception as e:
        print(f"Error consultando {nombre}: {e}")

Prometheus -> lambda26_bi_eventos_total: 481
Prometheus -> lambda26_bi_eventos_validos_ratio: 1
Prometheus -> lambda26_bi_latencia_promedio_ms: 2271.7151767151768


In [6]:
from pyspark.ml import PipelineModel
from pyspark.sql.types import StructType, StringType, DoubleType, LongType
import os

RUTA_MODELO_S10 = "./artifacts/models/modelo_ce_lr"
if not os.path.exists(RUTA_MODELO_S10):
    RUTA_MODELO_S10 = "../s10-series-tiempo-inferencia-streaming/artifacts/models/modelo_ce_lr"

modelo_cargado = PipelineModel.load(RUTA_MODELO_S10)
print("PipelineModel cargado correctamente desde:", RUTA_MODELO_S10)

esquema_evento = (
    StructType()
    .add("tipoEvento", StringType())
    .add("estacionId", StringType())
    .add("CE", DoubleType())
    .add("CM", DoubleType())
    .add("TempOut", DoubleType())
    .add("OutHum", DoubleType())
    .add("WindSpeed", DoubleType())
    .add("Bar", DoubleType())
    .add("SolarRad", DoubleType())
    .add("UVIndex", DoubleType())
    .add("origen", StringType())
    .add("timestamp", LongType())
)

crudo = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", "campo-electrico-eventos")
    .option("startingOffsets", "latest")
    .option("failOnDataLoss", "false")
    .load()
)

eventos_stream = (
    crudo.select(F.col("value").cast("string").alias("json_str"))
    .select(F.from_json(F.col("json_str"), esquema_evento).alias("e"))
    .select("e.*")
)

predicciones_stream = modelo_cargado.transform(eventos_stream)

PipelineModel cargado correctamente desde: ./artifacts/models/modelo_ce_lr


In [7]:
import shutil

RUTA_PREDICCIONES_STREAM = "./artifacts/predicciones_stream_s11"
CHECKPOINT_STREAM = "./artifacts/chk_predicciones_stream_s11"

shutil.rmtree(CHECKPOINT_STREAM, ignore_errors=True)
shutil.rmtree(RUTA_PREDICCIONES_STREAM, ignore_errors=True)

PUSHGATEWAY_BASE = "http://pushgateway:9091/metrics/job/lambda26_ml_streaming/instance"

def publicar_prediccion_en_vivo(estacion_id, ce_real, prediccion):
    cuerpo = (
        "# TYPE lambda26_ml_ce_real gauge\n"
        f"lambda26_ml_ce_real {ce_real}\n"
        "# TYPE lambda26_ml_prediccion_siguiente gauge\n"
        f"lambda26_ml_prediccion_siguiente {prediccion}\n"
    )
    peticion = urllib.request.Request(f"{PUSHGATEWAY_BASE}/{estacion_id}", data=cuerpo.encode("utf-8"), method="PUT")
    urllib.request.urlopen(peticion).close()

def guardar_predicciones_microlote(df_microlote, id_lote):
    filas = df_microlote.count()
    if filas == 0:
        return
    
    df_microlote.select(
        F.lit(id_lote).alias("batchId"),
        "estacionId",
        (F.col("timestamp") / 1000).cast("timestamp").alias("fechaHora"),
        "CE",
        F.col("prediction").alias("prediccionSiguienteCE")
    ).write.mode("append").parquet(RUTA_PREDICCIONES_STREAM)

    for fila in df_microlote.select("estacionId", "CE", "prediction").collect():
        publicar_prediccion_en_vivo(fila["estacionId"], fila["CE"], fila["prediction"])

    print(f"Micro-lote {id_lote}: {filas} prediccion(es) guardada(s) y publicada(s) al Pushgateway")

consulta_predicciones = (
    predicciones_stream.writeStream
    .foreachBatch(guardar_predicciones_microlote)
    .option("checkpointLocation", CHECKPOINT_STREAM)
    .start()
)

netlib-blas: JNI_OnLoad: dlopen(libblas.so.3) failed: libblas.so.3: cannot open shared object file: No such file or directory
                                                                                

Micro-lote 1: 1 prediccion(es) guardada(s) y publicada(s) al Pushgateway
Micro-lote 2: 1 prediccion(es) guardada(s) y publicada(s) al Pushgateway


Micro-lote 3: 1 prediccion(es) guardada(s) y publicada(s) al Pushgateway
Micro-lote 4: 1 prediccion(es) guardada(s) y publicada(s) al Pushgateway
Micro-lote 5: 1 prediccion(es) guardada(s) y publicada(s) al Pushgateway
Micro-lote 6: 1 prediccion(es) guardada(s) y publicada(s) al Pushgateway
Micro-lote 7: 1 prediccion(es) guardada(s) y publicada(s) al Pushgateway
Micro-lote 8: 1 prediccion(es) guardada(s) y publicada(s) al Pushgateway
Micro-lote 9: 1 prediccion(es) guardada(s) y publicada(s) al Pushgateway
Micro-lote 10: 1 prediccion(es) guardada(s) y publicada(s) al Pushgateway
Micro-lote 11: 1 prediccion(es) guardada(s) y publicada(s) al Pushgateway
Micro-lote 12: 1 prediccion(es) guardada(s) y publicada(s) al Pushgateway
Micro-lote 13: 1 prediccion(es) guardada(s) y publicada(s) al Pushgateway
Micro-lote 14: 1 prediccion(es) guardada(s) y publicada(s) al Pushgateway
Micro-lote 15: 1 prediccion(es) guardada(s) y publicada(s) al Pushgateway
Micro-lote 16: 1 prediccion(es) guardada(s) y

Micro-lote 32: 1 prediccion(es) guardada(s) y publicada(s) al Pushgateway
Micro-lote 33: 1 prediccion(es) guardada(s) y publicada(s) al Pushgateway
Micro-lote 34: 1 prediccion(es) guardada(s) y publicada(s) al Pushgateway
Micro-lote 35: 1 prediccion(es) guardada(s) y publicada(s) al Pushgateway
Micro-lote 36: 1 prediccion(es) guardada(s) y publicada(s) al Pushgateway
Micro-lote 37: 1 prediccion(es) guardada(s) y publicada(s) al Pushgateway


Micro-lote 38: 1 prediccion(es) guardada(s) y publicada(s) al Pushgateway
Micro-lote 39: 1 prediccion(es) guardada(s) y publicada(s) al Pushgateway
Micro-lote 40: 1 prediccion(es) guardada(s) y publicada(s) al Pushgateway


...

Micro-lote 50: 1 prediccion(es) guardada(s) y publicada(s) al Pushgateway


Micro-lote 51: 1 prediccion(es) guardada(s) y publicada(s) al Pushgateway


Micro-lote 52: 1 prediccion(es) guardada(s) y publicada(s) al Pushgateway


Micro-lote 53: 1 prediccion(es) guardada(s) y publicada(s) al Pushgateway
Micro-lote 54: 1 prediccion(es) guardada(s) y publicada(s) al Pushgateway


Micro-lote 55: 1 prediccion(es) guardada(s) y publicada(s) al Pushgateway
Micro-lote 56: 1 prediccion(es) guardada(s) y publicada(s) al Pushgateway
Micro-lote 57: 1 prediccion(es) guardada(s) y publicada(s) al Pushgateway


In [8]:
def consultar_prom(metrica):
    url = f"http://prometheus:9090/api/v1/query?query={metrica}"
    res = json.load(urllib.request.urlopen(url))
    datos = res["data"]["result"]
    return datos[0]["value"][1] if datos else None

primera = consultar_prom("lambda26_ml_ce_real")
print("Lectura inicial CE real:    ", primera)
print("Esperando 20 segundos a que lleguen nuevos micro-lotes...")
time.sleep(20)

segunda = consultar_prom("lambda26_ml_ce_real")
print("Lectura posterior CE real:  ", segunda)

if primera != segunda:
    print("\n¡Confirmado! La metrica cambio en vivo en Prometheus.")
else:
    print("\nValores identicos (espera un momento y vuelve a correrla).")

Lectura inicial CE real:     0.72
Esperando 20 segundos a que lleguen nuevos micro-lotes...
Micro-lote 58: 1 prediccion(es) guardada(s) y publicada(s) al Pushgateway
Micro-lote 59: 1 prediccion(es) guardada(s) y publicada(s) al Pushgateway
Micro-lote 60: 1 prediccion(es) guardada(s) y publicada(s) al Pushgateway
Micro-lote 61: 1 prediccion(es) guardada(s) y publicada(s) al Pushgateway
Micro-lote 62: 1 prediccion(es) guardada(s) y publicada(s) al Pushgateway
Micro-lote 63: 1 prediccion(es) guardada(s) y publicada(s) al Pushgateway
Lectura posterior CE real:   0.54

¡Confirmado! La metrica cambio en vivo en Prometheus.
Micro-lote 64: 1 prediccion(es) guardada(s) y publicada(s) al Pushgateway
Micro-lote 65: 1 prediccion(es) guardada(s) y publicada(s) al Pushgateway


Micro-lote 66: 1 prediccion(es) guardada(s) y publicada(s) al Pushgateway
Micro-lote 67: 1 prediccion(es) guardada(s) y publicada(s) al Pushgateway
Micro-lote 68: 1 prediccion(es) guardada(s) y publicada(s) al Pushgateway
Micro-lote 69: 1 prediccion(es) guardada(s) y publicada(s) al Pushgateway
Micro-lote 70: 1 prediccion(es) guardada(s) y publicada(s) al Pushgateway
Micro-lote 71: 1 prediccion(es) guardada(s) y publicada(s) al Pushgateway
Micro-lote 72: 1 prediccion(es) guardada(s) y publicada(s) al Pushgateway
Micro-lote 73: 1 prediccion(es) guardada(s) y publicada(s) al Pushgateway
Micro-lote 74: 1 prediccion(es) guardada(s) y publicada(s) al Pushgateway
Micro-lote 75: 1 prediccion(es) guardada(s) y publicada(s) al Pushgateway
Micro-lote 76: 1 prediccion(es) guardada(s) y publicada(s) al Pushgateway
Micro-lote 77: 1 prediccion(es) guardada(s) y publicada(s) al Pushgateway
Micro-lote 78: 1 prediccion(es) guardada(s) y publicada(s) al Pushgateway
Micro-lote 79: 1 prediccion(es) guarda

Micro-lote 103: 1 prediccion(es) guardada(s) y publicada(s) al Pushgateway


Micro-lote 104: 1 prediccion(es) guardada(s) y publicada(s) al Pushgateway


Micro-lote 105: 1 prediccion(es) guardada(s) y publicada(s) al Pushgateway
Micro-lote 106: 1 prediccion(es) guardada(s) y publicada(s) al Pushgateway
Micro-lote 107: 1 prediccion(es) guardada(s) y publicada(s) al Pushgateway
Micro-lote 108: 1 prediccion(es) guardada(s) y publicada(s) al Pushgateway
Micro-lote 109: 1 prediccion(es) guardada(s) y publicada(s) al Pushgateway


...

Micro-lote 542: 1 prediccion(es) guardada(s) y publicada(s) al Pushgateway


Micro-lote 543: 1 prediccion(es) guardada(s) y publicada(s) al Pushgateway


Micro-lote 544: 1 prediccion(es) guardada(s) y publicada(s) al Pushgateway
Micro-lote 545: 1 prediccion(es) guardada(s) y publicada(s) al Pushgateway
Micro-lote 546: 1 prediccion(es) guardada(s) y publicada(s) al Pushgateway


Micro-lote 547: 1 prediccion(es) guardada(s) y publicada(s) al Pushgateway


Micro-lote 548: 1 prediccion(es) guardada(s) y publicada(s) al Pushgateway
Micro-lote 549: 1 prediccion(es) guardada(s) y publicada(s) al Pushgateway
